In [ ]:
import numpy as np
from scipy.optimize import minimize
import matplotlib.pyplot as plt
import time

np.random.seed(0)

# =========================
# Part 2 (Revised version)
# Price update: follow Chapter 4 / Part 1 KKT rule
# Flow update : x^{t+1} = argmin_x sum_e (lambda_hat_e(x_e) - p_e^t)^2
# where lambda_hat_e(x_e) = alpha_e - beta_e * (a_e x_e + b_e)
# =========================

# === Step 1: Initialization ===
A = 2
iterations = 30
I_0 = 6
v = 1.35
noise_std = 0.05

# keep inner sample size under control
I_CAP = 300
P_MAX = 5000

# latency: c_e(x_e) = a_e x_e + b_e
a_e = np.array([3.0, 3.5])
b_e = np.array([0.1, 0.12])

# Initial values
prices = np.linspace(0.2, 0.6, A)
alpha = np.ones(A) * 3.0
beta  = np.ones(A) * 0.45

# History
alpha_hist, beta_hist, prices_hist, flows_hist = [], [], [], []
iteration_logs = []

# === Functions ===
def latency(x):
    return a_e * x + b_e

def compute_I_n(n):
    return int(min(I_CAP, max(I_0, I_0 * (v ** n))))

def compute_delta_n(I_n):
    return I_n ** (-1 / 4)

# === Step 2: Compute flows by solving
#     min_{x>=0, sum x=1} sum_e (lambda_hat_e(x_e) - p_e)^2
#     lambda_hat_e(x_e) = alpha_e - beta_e * (a_e x_e + b_e)
def update_flows_lambda(prices, alpha, beta, x_prev=None, maxiter=200, ftol=1e-8):
    def objective(x):
        c = latency(x)
        lambda_hat = alpha - beta * c
        return float(np.sum((lambda_hat - prices) ** 2))

    cons = ({"type": "eq", "fun": lambda x: np.sum(x) - 1.0},)
    bounds = [(0.0, 1.0) for _ in range(A)]

    if x_prev is None:
        x0 = np.ones(A) / A
    else:
        x0 = np.clip(np.array(x_prev, dtype=float), 0.0, 1.0)
        s = x0.sum()
        x0 = x0 / s if s > 0 else (np.ones(A) / A)

    res = minimize(
        objective,
        x0,
        method="SLSQP",
        bounds=bounds,
        constraints=cons,
        options={"ftol": float(ftol), "maxiter": int(maxiter), "disp": False}
    )

    if (not res.success) or (not np.isfinite(res.fun)) or (not np.all(np.isfinite(res.x))):
        x = x0
        ok = False
        msg = res.message
    else:
        x = res.x
        ok = True
        msg = res.message

    # numerical cleanup
    x = np.maximum(x, 0.0)
    s = x.sum()
    x = x / s if s > 0 else (np.ones(A) / A)

    return x, ok, msg, getattr(res, "nit", None)

# === Step 3: Parameter Estimation ===
def estimate_parameters(observed_lambda, t_values, prev_alpha, prev_beta):
    """
    Fit: lambda ≈ alpha - beta * t
    """
    def objective(params):
        a_i, b_i = params
        pred = a_i - b_i * t_values
        return float(np.sum((observed_lambda - pred) ** 2))

    result = minimize(objective, [prev_alpha, prev_beta])
    a_hat, b_hat = result.x

    # keep beta positive
    b_hat = max(float(b_hat), 1e-6)
    return float(a_hat), float(b_hat)

# === Step 4: Use Chapter 4 / Part 1 KKT update ===
def solve_kkt(alpha_i, beta_i, prices_i, i):
    sum_others = np.sum(prices_i) - prices_i[i]
    p = (alpha_i + beta_i * sum_others) / (2.0 * beta_i)
    return float(p)

# === CDL Iteration ===
flows_stage_prev = np.ones(A) / A

for n in range(iterations):
    I_n = compute_I_n(n)
    delta_n = compute_delta_n(I_n)

    # p_hat is the current reference price (keep this symbol only)
    p_hat = np.copy(prices)

    stage_t0 = time.time()
    slow_warn = False

    # update each road once per stage
    for m in range(A):
        i = m

        t_values = []
        observed_lambda = []

        # collect I_n samples for road i
        for t in range(I_n):
            prices_exp = np.copy(prices)

            # follow Chapter 4 style: experiment around current reference price
            prices_exp[i] = p_hat[i] + delta_n
            prices_exp = np.clip(prices_exp, -P_MAX, P_MAX)

            # flow from new lambda-based minimization
            flows, ok, msg, nit = update_flows_lambda(
                prices_exp, alpha, beta,
                x_prev=flows_stage_prev,
                maxiter=120,
                ftol=1e-8
            )
            flows_stage_prev = flows

            t_vec = latency(flows)
            t_values.append(t_vec[i])

            # simulated observation
            lam_obs = alpha[i] - beta[i] * t_vec[i] + np.random.normal(0, noise_std)
            observed_lambda.append(lam_obs)

            if (not ok) and (not slow_warn):
                print(f"[warn] n={n}, road={i}, SLSQP issue: {msg}")
                slow_warn = True

        # estimate alpha_i, beta_i
        alpha[i], beta[i] = estimate_parameters(
            np.array(observed_lambda),
            np.array(t_values),
            alpha[i], beta[i]
        )

        # IMPORTANT:
        # update price by Chapter 4 / Part 1 KKT rule
        prices[i] = solve_kkt(alpha[i], beta[i], prices, i)
        prices[i] = np.clip(prices[i], -P_MAX, P_MAX)

    # stage-end flow for logging
    flows, ok, msg, nit = update_flows_lambda(
        prices, alpha, beta,
        x_prev=flows_stage_prev,
        maxiter=200,
        ftol=1e-8
    )
    flows_stage_prev = flows

    alpha_hist.append(np.copy(alpha))
    beta_hist.append(np.copy(beta))
    prices_hist.append(np.copy(prices))
    flows_hist.append(np.copy(flows))

    elapsed = time.time() - stage_t0

    iteration_logs.append({
        "Iteration": n + 1,
        "I_n": I_n,
        "StageSec": round(elapsed, 3),
        "Alpha": np.round(alpha, 4),
        "Beta": np.round(beta, 4),
        "Prices": np.round(prices, 4),
        "Flows": np.round(flows, 4),
        "FlowOK": ok
    })

# === Convert to arrays ===
alpha_hist = np.array(alpha_hist)
beta_hist = np.array(beta_hist)
prices_hist = np.array(prices_hist)
flows_hist = np.array(flows_hist)

# === Logs ===
for log in iteration_logs:
    print(
        f"Iter {log['Iteration']:3d} | I_n={log['I_n']:3d} | sec={log['StageSec']:.3f} | "
        f"Alpha: {log['Alpha']} | Beta: {log['Beta']} | "
        f"Price: {log['Prices']} | Flow: {log['Flows']} | FlowOK={log['FlowOK']}"
    )

# === Visualization ===
fig, axs = plt.subplots(3, 2, figsize=(14, 12))

for i in range(A):
    axs[0, 0].plot(alpha_hist[:, i], label=f'Alpha {i+1}')
axs[0, 0].set_title("Alpha Evolution")
axs[0, 0].set_xlabel("Iteration")
axs[0, 0].set_ylabel("Alpha")
axs[0, 0].legend()
axs[0, 0].grid(True)

for i in range(A):
    axs[0, 1].plot(beta_hist[:, i], label=f'Beta {i+1}')
axs[0, 1].set_title("Beta Evolution")
axs[0, 1].set_xlabel("Iteration")
axs[0, 1].set_ylabel("Beta")
axs[0, 1].legend()
axs[0, 1].grid(True)

for i in range(A):
    axs[1, 0].plot(prices_hist[:, i], label=f'Price {i+1}')
axs[1, 0].set_title("Price Evolution")
axs[1, 0].set_xlabel("Iteration")
axs[1, 0].set_ylabel("Price")
axs[1, 0].legend()
axs[1, 0].grid(True)

for i in range(A):
    axs[1, 1].plot(flows_hist[:, i], label=f'Flow {i+1}')
axs[1, 1].set_title("Flow Evolution ")
axs[1, 1].set_xlabel("Iteration")
axs[1, 1].set_ylabel("Flow")
axs[1, 1].legend()
axs[1, 1].grid(True)

for i in range(A):
    axs[2, 0].plot(alpha_hist[:, i], beta_hist[:, i], label=f'Path {i+1}')
axs[2, 0].set_title("Alpha vs Beta")
axs[2, 0].set_xlabel("Alpha")
axs[2, 0].set_ylabel("Beta")
axs[2, 0].legend()
axs[2, 0].grid(True)

for i in range(A):
    axs[2, 1].scatter(prices_hist[:, i], flows_hist[:, i], alpha=0.6, label=f'Path {i+1}')
axs[2, 1].set_title("Price vs Flow")
axs[2, 1].set_xlabel("Price")
axs[2, 1].set_ylabel("Flow")
axs[2, 1].legend()
axs[2, 1].grid(True)

plt.tight_layout()
plt.show()
